# CSCI 4253 / 5253 - Lab #4 - Patent Problem with Spark DataFrames
<div>
 <h2> CSCI 4283 / 5253 
  <IMG SRC="https://www.colorado.edu/cs/profiles/express/themes/cuspirit/logo.png" WIDTH=50 ALIGN="right"/> </h2>
</div>

This [Spark cheatsheet](https://s3.amazonaws.com/assets.datacamp.com/blog_assets/PySpark_SQL_Cheat_Sheet_Python.pdf) is useful as is [this reference on doing joins in Spark dataframe](http://www.learnbymarketing.com/1100/pyspark-joins-by-example/).

The [DataBricks company has one of the better reference manuals for PySpark](https://docs.databricks.com/spark/latest/dataframes-datasets/index.html) -- they show you how to perform numerous common data operations such as joins, aggregation operations following `groupBy` and the like.

In [7]:
from pyspark import SparkContext, SparkConf
from pyspark.sql import SparkSession

The following aggregation functions may be useful -- [these can be used to aggregate results of `groupby` operations](https://docs.databricks.com/spark/latest/dataframes-datasets/introduction-to-dataframes-python.html#example-aggregations-using-agg-and-countdistinct). More documentation is at the [PySpark SQL Functions manual](https://spark.apache.org/docs/2.3.0/api/python/pyspark.sql.html#module-pyspark.sql.functions). Feel free to use other functions from that library.

In [8]:
from pyspark.sql.functions import col, count, countDistinct

Create our session as described in the tutorials

In [27]:
spark = SparkSession \
    .builder \
    .appName("Lab4-Dataframe") \
    .master("local[*]")\
    .getOrCreate()

Read in the citations and patents data and check that the data makes sense. Note that unlike in the RDD solution, the data is automatically inferred to be Integer() types.

In [28]:
citations = spark.read.load('cite75_99.txt.gz',
            format="csv", sep=",", header=True,
            compression="gzip",
            inferSchema="true")

Caching citations to avoid repeated loading

In [29]:
citations.cache()

DataFrame[CITING: int, CITED: int]

In [30]:
citations.show(6)

+-------+-------+
| CITING|  CITED|
+-------+-------+
|3858241| 956203|
|3858241|1324234|
|3858241|3398406|
|3858241|3557384|
|3858241|3634889|
|3858242|1515701|
+-------+-------+
only showing top 6 rows



In [31]:
patents = spark.read.load('apat63_99.txt.gz',
            format="csv", sep=",", header=True,
            compression="gzip",
            inferSchema="true")

Caching patents to avoid repeated loading

In [32]:
patents.cache()

DataFrame[PATENT: int, GYEAR: int, GDATE: int, APPYEAR: int, COUNTRY: string, POSTATE: string, ASSIGNEE: int, ASSCODE: int, CLAIMS: int, NCLASS: int, CAT: int, SUBCAT: int, CMADE: int, CRECEIVE: int, RATIOCIT: double, GENERAL: double, ORIGINAL: double, FWDAPLAG: double, BCKGTLAG: double, SELFCTUB: double, SELFCTLB: double, SECDUPBD: double, SECDLWBD: double]

In [33]:
patents.show(5)

+-------+-----+-----+-------+-------+-------+--------+-------+------+------+---+------+-----+--------+--------+-------+--------+--------+--------+--------+--------+--------+--------+
| PATENT|GYEAR|GDATE|APPYEAR|COUNTRY|POSTATE|ASSIGNEE|ASSCODE|CLAIMS|NCLASS|CAT|SUBCAT|CMADE|CRECEIVE|RATIOCIT|GENERAL|ORIGINAL|FWDAPLAG|BCKGTLAG|SELFCTUB|SELFCTLB|SECDUPBD|SECDLWBD|
+-------+-----+-----+-------+-------+-------+--------+-------+------+------+---+------+-----+--------+--------+-------+--------+--------+--------+--------+--------+--------+--------+
|3070801| 1963| 1096|   NULL|     BE|   NULL|    NULL|      1|  NULL|   269|  6|    69| NULL|       1|    NULL|    0.0|    NULL|    NULL|    NULL|    NULL|    NULL|    NULL|    NULL|
|3070802| 1963| 1096|   NULL|     US|     TX|    NULL|      1|  NULL|     2|  6|    63| NULL|       0|    NULL|   NULL|    NULL|    NULL|    NULL|    NULL|    NULL|    NULL|    NULL|
|3070803| 1963| 1096|   NULL|     US|     IL|    NULL|      1|  NULL|     2|  6|    6

### Creating an intermediate dataframe to store the STATE value in the citations table which can be used later in joins with other tables

In [73]:
citations_with_state = citations.join(patents, citations.CITED == patents.PATENT, "left") \
                                .select(citations.CITING, citations.CITED, patents.POSTATE.alias("STATE"))

In [74]:
citations_with_state.cache()

DataFrame[CITING: int, CITED: int, STATE: string]

In [61]:
citations_with_state.show(10)

+-------+-----+-----+
| CITING|CITED|STATE|
+-------+-----+-----+
|4192521| 2366| NULL|
|4253355| 2366| NULL|
|4305315| 2366| NULL|
|5580635| 5156| NULL|
|4976561| 5518| NULL|
|4480374| 5803| NULL|
|5123817| 6620| NULL|
|4115020| 7240| NULL|
|4727698| 7253| NULL|
|4108250| 7340| NULL|
+-------+-----+-----+
only showing top 10 rows



### Intermediate dataframe to store the `PATENT`, `CO_CITED_COUNT` columns

In [67]:
count_frame = (
    patents
    .join(
        citations_with_state,
        (patents.PATENT == citations_with_state.CITING) &
        (patents.POSTATE == citations_with_state.STATE) &
        patents.POSTATE.isNotNull(),
        "left"
    )
    .groupBy(patents.PATENT)
    .agg(
        count(citations_with_state.CITING).alias("CO_CITED_COUNT")
    )
    .orderBy("CO_CITED_COUNT", ascending=False)
)

In [68]:
count_frame.cache()

DataFrame[PATENT: int, CO_CITED_COUNT: bigint]

In [69]:
count_frame.show(10)

+-------+--------------+
| PATENT|CO_CITED_COUNT|
+-------+--------------+
|5959466|           125|
|5983822|           103|
|6008204|           100|
|5952345|            98|
|5998655|            96|
|5958954|            96|
|5936426|            94|
|5739256|            90|
|5951547|            90|
|5925042|            90|
+-------+--------------+
only showing top 10 rows



### Final output by left joining count_frame and patents table to get the full view of the table with an added `co_cited_count` column

In [75]:
final_output = (
    patents
    .join(count_frame, "PATENT", "left")
    .orderBy("CO_CITED_COUNT", ascending=False)
    .limit(13)
)

In [76]:
final_output.show()

+-------+-----+-----+-------+-------+-------+--------+-------+------+------+---+------+-----+--------+--------+-------+--------+--------+--------+--------+--------+--------+--------+--------------+
| PATENT|GYEAR|GDATE|APPYEAR|COUNTRY|POSTATE|ASSIGNEE|ASSCODE|CLAIMS|NCLASS|CAT|SUBCAT|CMADE|CRECEIVE|RATIOCIT|GENERAL|ORIGINAL|FWDAPLAG|BCKGTLAG|SELFCTUB|SELFCTLB|SECDUPBD|SECDLWBD|CO_CITED_COUNT|
+-------+-----+-----+-------+-------+-------+--------+-------+------+------+---+------+-----+--------+--------+-------+--------+--------+--------+--------+--------+--------+--------+--------------+
|5959466| 1999|14515|   1997|     US|     CA|    5310|      2|  NULL|   326|  4|    46|  159|       0|     1.0|   NULL|  0.6186|    NULL|  4.8868|  0.0455|   0.044|    NULL|    NULL|           125|
|5983822| 1999|14564|   1998|     US|     TX|  569900|      2|  NULL|   114|  5|    55|  200|       0|   0.995|   NULL|  0.7201|    NULL|   12.45|     0.0|     0.0|    NULL|    NULL|           103|
|6008204| 